Script to quantify error induced by using TMY weather with actual year load in a variety of locations with a variety of utility rate structures

Requires:
nrel-pysam
requests
numpy
pandas

In [54]:
import numpy as np
import pandas as pd
import json
import os
import csv

import PySAM.Battery as battery_model
import PySAM.Pvsamv1 as pv_model
import PySAM.Utilityrate5 as utility_rate
import PySAM.Cashloan as cashloan
import PySAM.ResourceTools
import PySAM.UtilityRateTools

In [55]:
import sys
print(sys.executable)

/Users/bspeetle/Desktop/repo/SAM-analyses/2025/battwatts_sensitivity/venv/bin/python


In [39]:
def get_pysam_json(json_file_path):
    """
    Open a PySAM JSON file and return as a dictionary
    """
    with open(json_file_path) as f:
        dic = json.load(f)
    return dic


def get_exact_shading_keys(json_file_path, desired_keys):

    with open(json_file_path, "r") as f:
        data = json.load(f)

    return {k: data[k] for k in desired_keys if k in data}

"""
 - Original get load profile for ResStock CSVs, not postprocessed csvs from locally run analysis
def get_load_profile(load_path, desired_timestep_minutes):
    
    Get data out of a CSV file
    Original ResStock data is 15 minute kWh data - need hourly for TMY comparison and to convert to kW
    
    df = pd.read_csv(load_path)
    timeseries = df["out.electricity.total.energy_consumption"].values
    
    if desired_timestep_minutes < 15:
        raise ValueError("get_load_profile is not set up for timesteps less than 15 minutes.")
    elif desired_timestep_minutes == 15:
        return timeseries * 4 # Convert from kWh to kW
    elif desired_timestep_minutes % 15 != 0: 
        raise ValueError("get_load_profile is not set up for that aren't evenly divisiable by 15. Pick 15, 30, or 60.")
    else:
        averaged_timesteps = []
        steps_per_step = desired_timestep_minutes / 15
        steps_per_hr = 60 / desired_timestep_minutes
        step = 0
        avg_kwhs = 0
        for kwh in timeseries:
             avg_kwhs += kwh
             step += 1
             if step == steps_per_step:
                 averaged_timesteps.append(avg_kwhs * steps_per_hr)
                 step = 0
                 avg_kwhs = 0
        return averaged_timesteps
"""
def get_load_profile(load_path):

    df = pd.read_csv(load_path)
    timeseries = df["kW"].values

    return timeseries

In [50]:
def run_location_withbattery(weather_file, rate_path, load_path, shade_path, nameplate_loss):

    # Utility rate data is contained here since the battery needs it for dispatch calculations
    rate_setup = get_pysam_json(rate_path)
    wf_timestep = 60
    #load_profile = get_load_profile(load_path, wf_timestep)
    load_profile = get_load_profile(load_path)
    keys_to_include = ["subarray1_shading_diff","subarray1_shading_en_diff","subarray1_shading_en_timestep","subarray1_shading_timestep"]
    shade_setup = get_exact_shading_keys(shade_path,keys_to_include)

    pv = pv_model.default("PVBatteryResidential") # PV with Battery: "PVBatteryResidential". PV only: "FlatPlatePVResidential"
    ur = utility_rate.from_existing(pv, "PVBatteryResidential")
    cl = cashloan.from_existing(ur, "PVBatteryResidential")

    if nameplate_loss == 0:
        for k in keys_to_include:
            value = shade_setup[k]
            pv.value(k, value)
    else:
        pv.value("subarray1_nameplate_loss", nameplate_loss) 


    for k, v in rate_setup.items():
        try:
            pv.value(k, v)
        except AttributeError:
            if "batt_adjust" in k:
                pass
            else:
                print("Failed to assign PV key " + str(k))

    pv.value("solar_resource_file", str(weather_file))
    pv.value("batt_dispatch_choice", 4) # Retail rates dispatch
    pv.value("load", load_profile)
    pv.value("en_batt", 1)
    cl.value("en_batt",1)

    pv.execute()
    ur.execute()
    cl.execute()

    output_data = {}
    output_data.update(pv.Outputs.export())
    output_data.update(ur.Outputs.export())
    output_data.update(cl.Outputs.export())
    return output_data


In [58]:
def run_location_pvonly(weather_file, rate_path, load_path, shade_path, nameplate_loss):

    # Utility rate data is contained here since the battery needs it for dispatch calculations
    rate_setup = get_pysam_json(rate_path)
    wf_timestep = 60
    #load_profile = get_load_profile(load_path, wf_timestep)
    load_profile = get_load_profile(load_path)
    keys_to_include = ["subarray1_shading_diff","subarray1_shading_en_diff","subarray1_shading_en_timestep","subarray1_shading_timestep"]
    shade_setup = get_exact_shading_keys(shade_path,keys_to_include)

    pv = pv_model.default("FlatPlatePVResidential") # PV with Battery: "PVBatteryResidential". PV only: "FlatPlatePVResidential"
    ur = utility_rate.from_existing(pv, "FlatPlatePVResidential")
    cl = cashloan.from_existing(ur, "FlatPlatePVResidential")

    if nameplate_loss == 0:
        for k in keys_to_include:
            value = shade_setup[k]
            pv.value(k, value)
    else:
        pv.value("subarray1_nameplate_loss", nameplate_loss) 


    for k, v in rate_setup.items():
        try:
            pv.value(k, v)
        except AttributeError:
            if "batt_adjust" in k:
                pass
            else:
                print("Failed to assign PV key " + str(k))

    pv.value("solar_resource_file", str(weather_file))
    pv.value("batt_dispatch_choice", 4) # Retail rates dispatch
    pv.value("load", load_profile)
    pv.value("en_batt", 0)
    cl.value("en_batt",0)

    pv.execute()
    ur.execute()
    cl.execute()

    output_data = {}
    output_data.update(pv.Outputs.export())
    output_data.update(ur.Outputs.export())
    output_data.update(cl.Outputs.export())
    return output_data


In [59]:
#Test for one instance
file_dir = os.path.abspath('') #ensure this is ""Users/bspeetle/Desktop/repo/SAM-analyses/2025/battwatts_sensitivity/"
print(file_dir)
weather_path = file_dir + "/weather_data/AZ_35.14_-111.67_35.14_-111.67_nsrdb-GOES-aggregated-v4-0-0_60_2018.csv"
rate_path = file_dir + "/rate_data/" + "arizona" + "_" + "aps" + "_2025_pvsamv1.json"
load_path = file_dir + "/load_data/" + "load-data-AZ-112157_hourly.csv"
shade_path = file_dir + "/shade_files/3D_Shade_Tool_AMY_2018_pvsamv1.json"
test_output = run_location_pvonly(weather_path, rate_path, load_path, shade_path, 5.902)
#So it runs for the withbattery_ version, there is something else that needs to change for the pvonly_ version. 

/Users/bspeetle/Desktop/repo/SAM-analyses/2025/battwatts_sensitivity


To generate rate data:

- Open the SAM file in sam files
- Download the rate data on the utility rates page
- Make adjustments as needed (e.g. "net billing" for California rates)
- Use shift-F5 to export the code and choose "PySAM JSON"
- Save the untitled_pvsamv1.json file to the rate_data folder and rename it to something more descriptive

In [9]:
print(len(test_output))

526


In [68]:
file_dir = os.path.abspath('') #ensure this is ""Users/bspeetle/Desktop/repo/SAM-analyses/2025/battwatts_sensitivity/"

state_long = ["arizona","california","colorado"] 
utility_names = ["aps","sdge","xcel"]
bd_id = "112157"
lat = "35.14"
lon = "-111.67"
years = [2018, 2019, 2020, 2021, 2022, 2023]
weather_path_tmy = file_dir + "/weather_data/" + "AZ" + "_" + lat + "_" + lon + "_" + lat + "_" + lon + "_nsrdb-GOES-tmy-v4-0-0_60_tmy.csv"
shade_path_tmy = file_dir + "/shade_files/" + "3D_Shade_Tool_TMY_pvsamv1.json"
#load_path_tmy = file_dir + "/load_data/" + "load-data-AZ-112157_hourly.csv"
all_results = []
nameplate_loss_amy = [5.9092, 5.4976, 6.2223, 6.1837, 6.1568, 5.7723]
nameplate_loss_tmy = 5.9619
building_path = file_dir + "/building_models/bldg" + bd_id + "-up00/"

for st_long, utility in zip(
    state_long, utility_names
):
    rate_path = file_dir + "/rate_data/" + st_long + "_" + utility + "_2025_pvsamv1.json"
    
    for year in years:
        load_path = building_path + str(year) + "-" + bd_id + "_kw_load.csv"
        weather_path_amy = building_path + "weather_data/" + 'nsrdb_' + lat + '_' + lon + '_nsrdb-GOES-aggregated-v4-0-0_60_' + str(year) + ".csv"
        shade_path_amy = file_dir + "/shade_files/" + "3D_Shade_Tool_AMY_" + str(year) + "_pvsamv1.json"

        #Need to revise this message:

        print(f"Running AMY analysis for {st_long}, building ID {bd_id}, year{year}, PV only...")
        amy_outputs_pvonly_anloss = run_location_pvonly(weather_path_amy, rate_path, load_path, shade_path_amy,nameplate_loss_amy[years.index(year)])
        
        amy_outputs_pvonly_3dshade = run_location_pvonly(weather_path_amy, rate_path, load_path, shade_path_amy,0)

        print(f"Running AMY analysis for {st_long}, building ID {bd_id}, year{year}, with battery...")
        
        amy_outputs_withbattery_anloss = run_location_withbattery(weather_path_amy, rate_path, load_path, shade_path_amy,nameplate_loss_amy[years.index(year)])

        amy_outputs_withbattery_3dshade = run_location_withbattery(weather_path_amy, rate_path, load_path, shade_path_amy,0)

        annual_bills_amy_pvonly_anloss = amy_outputs_pvonly_anloss.get("utility_bill_w_sys", [None]*12)
        annual_bills_amy_pvonly_3dshade = amy_outputs_pvonly_3dshade.get("utility_bill_w_sys", [None]*12)
        annual_bills_amy_withbattery_anloss = amy_outputs_withbattery_anloss.get("utility_bill_w_sys", [None]*12)
        annual_bills_amy_withbattery_3dshade = amy_outputs_withbattery_3dshade.get("utility_bill_w_sys", [None]*12)

        print(f"Running TMY analysis for {st_long}, building ID {bd_id}, load year{year}, PV only...")

        tmy_outputs_pvonly_anloss = run_location_pvonly(weather_path_tmy, rate_path, load_path, shade_path_tmy,nameplate_loss_tmy)

        tmy_outputs_pvonly_3dshade = run_location_pvonly(weather_path_tmy, rate_path, load_path, shade_path_tmy,0)

        print(f"Running TMY analysis for {st_long}, building ID {bd_id}, load year{year}, with Battery...")


        tmy_outputs_withbattery_anloss = run_location_withbattery(weather_path_tmy, rate_path, load_path, shade_path_tmy,nameplate_loss_tmy)

        tmy_outputs_withbattery_3dshade = run_location_withbattery(weather_path_tmy, rate_path, load_path, shade_path_tmy,0)
    
        annual_bills_tmy_pvonly_anloss = tmy_outputs_pvonly_anloss.get("utility_bill_w_sys", [None]*12)
        annual_bills_tmy_pvonly_3dshade = tmy_outputs_pvonly_3dshade.get("utility_bill_w_sys", [None]*12)
        annual_bills_tmy_withbattery_anloss = tmy_outputs_withbattery_anloss.get("utility_bill_w_sys", [None]*12)
        annual_bills_tmy_withbattery_3dshade = tmy_outputs_withbattery_3dshade.get("utility_bill_w_sys", [None]*12)

        # Create the filtered output for TMY
        filtered_tmy = {
            "savings_year1_pvonly_anloss": tmy_outputs_pvonly_anloss.get("savings_year1", 0),
            "annual_energy_pvonly_anloss": tmy_outputs_pvonly_anloss.get("annual_energy", 0),
            "year_1_bill_pvonly_anloss": annual_bills_tmy_pvonly_anloss[1],
            "savings_year1_withbattery_anloss": tmy_outputs_withbattery_anloss.get("savings_year1", 0),
            "annual_energy_withbattery_anloss": tmy_outputs_withbattery_anloss.get("annual_energy", 0),
            "year_1_bill_withbattery_anloss": annual_bills_tmy_withbattery_anloss[1],

            "savings_year1_pvonly_3dshade": tmy_outputs_pvonly_3dshade.get("savings_year1", 0),
            "annual_energy_pvonly_3dshade": tmy_outputs_pvonly_3dshade.get("annual_energy", 0),
            "year_1_bill_pvonly_3dshade": annual_bills_tmy_pvonly_3dshade[1],
            "savings_year1_withbattery_3dshade": tmy_outputs_withbattery_3dshade.get("savings_year1", 0),
            "annual_energy_withbattery_3dshade": tmy_outputs_withbattery_3dshade.get("annual_energy", 0),
            "year_1_bill_withbattery_3dshade": annual_bills_tmy_withbattery_3dshade[1]
        }

        # Create the filtered output for AMY (same logic)
        filtered_amy = {
            "savings_year1_pvonly_anloss": amy_outputs_pvonly_anloss.get("savings_year1"),
            "annual_energy_pvonly_anloss": amy_outputs_pvonly_anloss.get("annual_energy"),
            "year_1_bill_pvonly_anloss": annual_bills_amy_pvonly_anloss[1],
            "savings_year1_withbattery_anloss": amy_outputs_withbattery_anloss.get("savings_year1"),
            "annual_energy_withbattery_anloss": amy_outputs_withbattery_anloss.get("annual_energy"),
            "year_1_bill_withbattery_anloss": annual_bills_amy_withbattery_anloss[1],
            
            "savings_year1_pvonly_3dshade": amy_outputs_pvonly_3dshade.get("savings_year1"),
            "annual_energy_pvonly_3dshade": amy_outputs_pvonly_3dshade.get("annual_energy"),
            "year_1_bill_pvonly_3dshade": annual_bills_amy_pvonly_3dshade[1],
            "savings_year1_withbattery_3dshade": amy_outputs_withbattery_3dshade.get("savings_year1"),
            "annual_energy_withbattery_3dshade": amy_outputs_withbattery_3dshade.get("annual_energy"),
            "year_1_bill_withbattery_3dshade": annual_bills_amy_withbattery_3dshade[1]

        }



        result_row = {
            "state_long": st_long,
            "utility": utility,
            "building_id": bd_id,
            "latitude": lat,
            "longitude": lon,
            "year" : year,
            "run_type": "tmy", 
            **filtered_tmy  
        }
        all_results.append(result_row)

        result_row = {
            "state_long": st_long,
            "utility": utility,
            "building_id": bd_id,
            "latitude": lat,
            "longitude": lon,
            "year" : year,
            "run_type": "amy",
            **filtered_amy
        }
        all_results.append(result_row)

Running AMY analysis for arizona, building ID 112157, year2018, PV only...
Running AMY analysis for arizona, building ID 112157, year2018, with battery...
Running TMY analysis for arizona, building ID 112157, load year2018, PV only...
Running TMY analysis for arizona, building ID 112157, load year2018, with Battery...
Running AMY analysis for arizona, building ID 112157, year2019, PV only...
Running AMY analysis for arizona, building ID 112157, year2019, with battery...
Running TMY analysis for arizona, building ID 112157, load year2019, PV only...
Running TMY analysis for arizona, building ID 112157, load year2019, with Battery...
Running AMY analysis for arizona, building ID 112157, year2020, PV only...
Running AMY analysis for arizona, building ID 112157, year2020, with battery...
Running TMY analysis for arizona, building ID 112157, load year2020, PV only...
Running TMY analysis for arizona, building ID 112157, load year2020, with Battery...
Running AMY analysis for arizona, buildi

In [69]:
output_file = "battwatts_sensitivity_outputs_shade_todiscuss.csv"
keys = all_results[0].keys()  # use first result's keys as headers

with open(output_file, 'w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=keys)
    writer.writeheader()
    writer.writerows(all_results)

print(f"Saved {len(all_results)} rows to {output_file}")
print(all_results)

#print(tmy_outputs["utility_bill_w_sys"][1])
#print(tmy_outputs["savings_year1"])
#print(tmy_outputs["annual_energy"])
#print(tmy_outputs["npv"])

#print(amy_outputs["utility_bill_w_sys"][1])
#print(amy_outputs["savings_year1"])
#print(amy_outputs["annual_energy"])
#print(amy_outputs["npv"])

#print("Energy difference " + str(amy_outputs["annual_energy"] / tmy_outputs["annual_energy"]) )
#print("Bill difference " + str(amy_outputs["savings_year1"] / tmy_outputs["savings_year1"]))




Saved 36 rows to battwatts_sensitivity_outputs_shade_todiscuss.csv
[{'state_long': 'arizona', 'utility': 'aps', 'building_id': '112157', 'latitude': '35.14', 'longitude': '-111.67', 'year': 2018, 'run_type': 'tmy', 'savings_year1_pvonly_anloss': 924.939431974622, 'annual_energy_pvonly_anloss': 11330.22237422229, 'year_1_bill_pvonly_anloss': 705.9252528552471, 'savings_year1_withbattery_anloss': 1117.537322452467, 'annual_energy_withbattery_anloss': 11106.883639154366, 'year_1_bill_withbattery_anloss': 513.3273623774065, 'savings_year1_pvonly_3dshade': 924.939431974622, 'annual_energy_pvonly_3dshade': 11330.22237422229, 'year_1_bill_pvonly_3dshade': 705.9252528552471, 'savings_year1_withbattery_3dshade': 1117.537322452467, 'annual_energy_withbattery_3dshade': 11106.883639154366, 'year_1_bill_withbattery_3dshade': 513.3273623774065}, {'state_long': 'arizona', 'utility': 'aps', 'building_id': '112157', 'latitude': '35.14', 'longitude': '-111.67', 'year': 2018, 'run_type': 'amy', 'savings_

In [ ]:
print(type(amy_outputs))
print(len(str(tmy_outputs)))